# D2-06 Variability and uncertainty in regionalised LCIA

Day 1 focused on uncertainty attached to inventories and exchanges. Here, we focus instead on variability and uncertainty in the **characterization factors themselves**.


## Learning goals

After this notebook, you should be able to:

- distinguish Day 1 inventory uncertainty from Day 2 CF variability
- inspect how uncertainty is encoded in local `edges` JSON files
- run a simple Monte Carlo exercise on regionalized CFs
- interpret score spread, overlap, and ranking robustness with boxplots and simple probabilities


## Background references

- Sacchi, R., Menacho, A. H., Seitfudem, G., Agez, M., Schlesinger-Martinat, J., Koyamparambath, A., Saldivar, J. S., Loubet, P., & Bauer, C. (2025). Contextual LCIA without the overhead: an exchange-based framework for flexible impact assessment. *The International Journal of Life Cycle Assessment, 30*(12), 3087-3101. https://doi.org/10.1007/s11367-025-02551-7
- Seitfudem, G., Berger, M., Schmied, H. M., & Boulay, A. (2025). The updated and improved method for water scarcity impact assessment in LCA, AWARE2.0. *Journal of Industrial Ecology, 29*(3), 891-907. https://doi.org/10.1111/jiec.70023


## 1) What changes relative to Day 1?

Day 1 question: "What if the inventory exchanges themselves are uncertain?"

Day 2 question: "What if the characterization factor applied to an exchange varies because the exact regional context is not fully fixed?"


## 2) Inspect two bundled `edges` examples with uncertainty blocks

`lcia_example_5.json` and `lcia_example_6.json` illustrate two styles:

- standard continuous uncertainty distributions
- nested discrete-plus-continuous uncertainty structures


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

example_5_path = Path('/Users/romain/GitHub/barcelona-regionalized-lcia/tutorials/DAY 2/assets/edges_examples/lcia_example_5.json')
example_6_path = Path('/Users/romain/GitHub/barcelona-regionalized-lcia/tutorials/DAY 2/assets/edges_examples/lcia_example_6.json')


def summarize_uncertain_method(path):
    data = json.loads(path.read_text())
    return pd.DataFrame([
        {
            'file': path.name,
            'flow': exc['supplier'].get('name'),
            'consumer matrix': exc['consumer'].get('matrix'),
            'base value': exc.get('value'),
            'distribution': exc.get('uncertainty', {}).get('distribution'),
        }
        for exc in data['exchanges']
    ])

pd.concat([
    summarize_uncertain_method(example_5_path),
    summarize_uncertain_method(example_6_path),
], ignore_index=True)


## 3) Build a small Monte Carlo experiment on regionalized CFs

We compare two stylized electricity mixes. The inventory is fixed; only the regionalized CFs vary.


In [ ]:
mixes = pd.DataFrame({
    'Spain low-voltage mix': {
        'stressed basin withdrawal': 0.22,
        'moderate basin withdrawal': 0.08,
        'hydropower reservoir withdrawal': 0.04,
    },
    'France low-voltage mix': {
        'stressed basin withdrawal': 0.05,
        'moderate basin withdrawal': 0.12,
        'hydropower reservoir withdrawal': 0.03,
    },
}).fillna(0.0)

mixes


In [ ]:
rng = np.random.default_rng(42)
iterations = 2000

cf_samples = {
    'stressed basin withdrawal': np.clip(rng.normal(loc=70, scale=12, size=iterations), a_min=20, a_max=None),
    'moderate basin withdrawal': np.clip(rng.normal(loc=28, scale=6, size=iterations), a_min=5, a_max=None),
    'hydropower reservoir withdrawal': np.clip(rng.triangular(left=2, mode=6, right=12, size=iterations), a_min=0, a_max=None),
}

scores = {}
for mix_name in mixes.columns:
    score = np.zeros(iterations)
    for flow_name, amount in mixes[mix_name].items():
        score += amount * cf_samples[flow_name]
    scores[mix_name] = score

scores_df = pd.DataFrame(scores)
scores_df.head()


## 4) Visualize the spread

Boxplots are a compact way to communicate variability in regionalized LCIA results.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.boxplot([scores_df[col] for col in scores_df.columns], labels=scores_df.columns)
ax.set_ylabel('Regionalized score')
ax.set_title('Monte Carlo on regionalized characterization factors')
plt.xticks(rotation=12)
plt.tight_layout()
plt.show()


## 5) Summarize overlap and ranking robustness

We use percentiles and a simple probability-of-superiority metric.


In [ ]:
summary = pd.DataFrame({
    'mean': scores_df.mean(),
    'p05': scores_df.quantile(0.05),
    'p50': scores_df.quantile(0.50),
    'p95': scores_df.quantile(0.95),
})
summary


In [ ]:
probability_france_lower = float((scores_df['France low-voltage mix'] < scores_df['Spain low-voltage mix']).mean())
print('P(France score < Spain score) =', round(probability_france_lower, 3))


## Checkpoint 1

Interpret the boxplot and the probability-of-superiority result:

- Which mix tends to score lower?
- Is the ranking perfectly certain?
- What part of the model is varying here?


In [ ]:
# TODO
interpretation = {
    'lower_scoring_mix': '',
    'ranking_is_certain': '',
    'what_varies': '',
}
interpretation


In [ ]:
interpretation = {
    'lower_scoring_mix': 'France low-voltage mix tends to score lower in this toy example.',
    'ranking_is_certain': 'No. The boxplots and the probability below 1.0 show overlap and therefore some ranking instability.',
    'what_varies': 'Only the regionalized characterization factors vary here; the inventory stays fixed.',
}
interpretation


## Key caution for practitioners

A change in LCIA ranking can come from regionalized CF variability even when the underlying inventory does not change. That is why Day 2 focuses on communicating spread and ranking robustness, not only point estimates.


## Recap

After this notebook, you should now be able to:

- explain the difference between inventory uncertainty and CF variability
- inspect uncertainty information in local `edges` method files
- run a small Monte Carlo on regionalized CFs
- use boxplots and simple probabilities to discuss ranking robustness
